# OAK-D Lite 현장 점검 노트북

연습 때 **OAK 상태와 링 버퍼 표본**을 확인하는 노트북이다. 파일과 ROS CLI(읽기 전용)만 쓴다 — 구동·짐벌·발사 명령을 보내는 코드는 없다.

- 출력은 **비운 채로 저장**한다(경기장 이미지가 저장소에 남지 않게. 대회 자료는 외부 공유 금지). 저장 전에 `Kernel → Restart Kernel and Clear Outputs`.
- 위에서부터 순서대로 실행한다. `[실기 확인]` 표시는 판·장치마다 다를 수 있는 값이다.

## 1. 설정 — 지금 설정의 `oak.*`, `oak_det.*`, `depth.*`

In [ ]:
import dataclasses
import json
import os
import pprint
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for _p in [ROOT, *ROOT.parents]:
    if (_p / "maicon26").is_dir():
        ROOT = _p
        break
sys.path.insert(0, str(ROOT))

from maicon26.launch_profile import load_run_config
from maicon26.validate import run_log_dir

CONFIG = os.environ.get("MAICON_CONFIG") or str(ROOT / "config" / "waveshare_real.json")
OVERRIDES = ROOT / "config" / "field_overrides.json"
cfg, applied = load_run_config(CONFIG, str(OVERRIDES) if OVERRIDES.exists() else None, "nav2", root=ROOT)
print("설정 파일:", CONFIG, "| 적용한 덮어쓰기:", len(applied))
for name in ("oak", "oak_det", "depth"):
    print(f"--- {name}")
    pprint.pprint(dataclasses.asdict(getattr(cfg, name)), sort_dicts=False)

## 2. 토픽 — 드라이버가 실제로 내보내는 이름과 주기

토픽·노드 이름은 판마다 다르다 `[실기 확인]`. 위 설정의 이름과 다르면 `oak.rgb_topic` / `oak.rgb_info_topic` 을 고친다.

In [ ]:
rgb_topic, info_topic = cfg.oak.rgb_topic, cfg.oak.rgb_info_topic
OAK_NODE = "/oak"        # [실기 확인] ros2 node list

In [ ]:
!ros2 topic list | grep oak

In [ ]:
!timeout 10 ros2 topic hz {rgb_topic}

In [ ]:
!ros2 topic echo --once {info_topic}

In [ ]:
!ros2 param dump {OAK_NODE}

## 3. 링 버퍼 — 최근 표본에 JSON 의 상자를 겹쳐 본다

영상은 BGR 로 저장돼 있다. **화면에 그리려고 RGB 로 바꾸는 것은 이 셀뿐**이다(탐지 입력과는 무관).

In [ ]:
import cv2
import matplotlib.pyplot as plt

save_dir = Path(cfg.oak_det.save_dir).expanduser() if cfg.oak_det.save_dir else run_log_dir() / "oak_samples"
samples = sorted(save_dir.glob("oak_*.jpg"))
print("표본 폴더:", save_dir, "| 장수:", len(samples))
show = samples[-6:]
if show:
    fig, axes = plt.subplots(1, len(show), figsize=(4.5 * len(show), 3.5), squeeze=False)
    for ax, jpg in zip(axes[0], show):
        meta_path = jpg.with_suffix(".json")
        meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
        bgr = cv2.imread(str(jpg))                                    # BGR
        for b in meta.get("boxes", []):
            x1, y1, x2, y2 = [int(v) for v in b["box"]]
            cv2.rectangle(bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(bgr, f"{b['cls']} {b['conf']:.2f}", (x1, max(y1 - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))                # 표시 전용 변환
        ax.set_title(f"{meta.get('why', '?')} | {meta.get('source', '?')}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 4. 유효성 통계 — `validation_status.json`

버린 데이터(규격 밖 값)의 출처별 개수와 마지막으로 유효했던 시각. 로봇이 실행 중일 때 2 s 마다 갱신된다.

In [ ]:
status_path = run_log_dir() / "validation_status.json"
if not status_path.exists():
    print("아직 없다:", status_path, "— 로봇 프로그램이 실행 중인지, MAICON_LOG_DIR 이 같은지 확인")
else:
    status = json.loads(status_path.read_text(encoding="utf-8"))
    rows = [(src, d["dropped"], d.get("last_reason") or "-", d.get("valid_count", 0),
             "-" if d.get("last_valid_age_s") is None else f"{d['last_valid_age_s']:.1f}")
            for src, d in sorted(status.get("sources", {}).items())]
    header = ("출처", "버린 수", "마지막 이유", "정상 수", "마지막 정상 후(s)")
    try:
        import pandas as pd
        display(pd.DataFrame(rows, columns=header))
    except ImportError:
        print(" | ".join(header))
        for r in rows:
            print(" | ".join(str(c) for c in r))
    print("기록 시각(벽시계):", status.get("written_wall"), "| 가동(s):", round(status.get("uptime_s", 0.0), 1))

## 5. 깊이 (RGBD 프로파일일 때만)

줄자로 잰 거리와 깊이 영상 중앙 영역의 거리를 나란히 적는다. 값은 직접 채운다. `읽은 값`은 깊이 영상에서 중앙 영역의 중앙값(m)이다.

측정 방법은 현장에서 정한다 `[실기 확인]`: 예) `ros2 topic echo --once --field data` 의 결과를 파일로 받아 읽거나, 별도 뷰어의 값을 적는다.

In [ ]:
if cfg.oak.profile != "rgbd":
    print("oak.profile =", cfg.oak.profile, "→ 깊이 점검은 건너뛴다 (RGBD 프로파일일 때만)")
else:
    # (줄자 m, 읽은 값 m) — 읽은 값을 현장에서 채운다
    table = [(0.5, None), (1.0, None), (1.5, None), (2.0, None), (3.0, None)]
    print("줄자(m) | 읽은 값(m) | 오차(m) | 오차(%)")
    for tape, read in table:
        if read is None:
            print(f"{tape:6.2f} | {'(적기)':>9} |")
        else:
            print(f"{tape:6.2f} | {read:9.3f} | {read - tape:+7.3f} | {100 * (read - tape) / tape:+6.1f}")
    print("참고: depth.min_range =", cfg.depth.min_range, "m, depth.max_range =", cfg.depth.max_range, "m")

## 6. 정리 확인 — 표본 파일 수가 `save_max` 이하인가

In [ ]:
import re

limit = max(0, min(int(cfg.oak_det.save_max), 10))
pat = re.compile(r"^oak_\d{6}_\d+\.(jpg|json)$")
mine = [p for p in save_dir.glob("oak_*") if pat.match(p.name)] if save_dir.exists() else []
jpgs = [p for p in mine if p.suffix == ".jpg"]
print(f"jpg {len(jpgs)}장 / 한도 {limit}장 (oak_det.save_max={cfg.oak_det.save_max}, 최대 10)")
print("OK" if len(jpgs) <= limit else "WARN: 한도를 넘었다 — 쓰기·정리 스레드가 돌고 있는지 확인")